In [4]:
import numpy as np

In [2]:
class ELM:
    def __init__(self, n_inputs, n_hidden, n_outputs, seed=42):
        rng = np.random.default_rng(seed)
        # Random, FIXED input-to-hidden weights and biases (never trained)
        self.W = rng.uniform(-1, 1, size=(n_inputs, n_hidden))
        self.b = rng.uniform(-1, 1, size=(n_hidden,))
        self.beta = None  # hidden-to-output weights, solved analytically

    def _hidden_activations(self, X):
        return self._sigmoid(X @ self.W + self.b)

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, Y):
        H = self._hidden_activations(X)          # (n_samples, n_hidden)
        # beta = H^+ * Y  (Moore-Penrose pseudoinverse -> least squares solution)
        self.beta = np.linalg.pinv(H) @ Y

    def predict(self, X):
        H = self._hidden_activations(X)
        return H @ self.beta


In [5]:
if __name__ == "__main__":
    rng = np.random.default_rng(0)

    # ---- 1. Create synthetic data: 6 inputs -> 3 outputs ----
    n_samples = 200
    X = rng.uniform(-1, 1, size=(n_samples, 6))

    # Some arbitrary nonlinear relationship, just for demonstration
    Y = np.column_stack([
        np.sin(X[:, 0]) + X[:, 1] * X[:, 2],
        X[:, 3]**2 - X[:, 4],
        np.cos(X[:, 5]) + X[:, 0] * X[:, 3],
    ])

    # ---- 2. Split train/test ----
    split = 150
    X_train, X_test = X[:split], X[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    # ---- 3. Build and train ELM ----
    model = ELM(n_inputs=6, n_hidden=50, n_outputs=3)
    model.fit(X_train, Y_train)          # single-step, no iterations

    # ---- 4. Evaluate ----
    Y_pred = model.predict(X_test)
    mse = np.mean((Y_pred - Y_test) ** 2)
    print(f"Test MSE: {mse:.5f}")
    print("Sample prediction:", Y_pred[0])
    print("Sample true value:", Y_test[0])

Test MSE: 0.02415
Sample prediction: [ 0.68145382 -0.3386687   0.57645386]
Sample true value: [ 0.85733113 -0.42178765  0.57579798]
